# 00 — Environment tests and GPU pilot

**Purpose:** Check free GPU access, local safety tests, packages, tokenizer alignment, and optional 2-step LoRA/full timing pilots.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# GitHub setup: run this as the FIRST code cell in Kaggle or Colab.
# This clones the source code only; GPU checkpoints/results need separate storage.
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
REPO_FOLDER = "nlp_research_proj"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # True: update an existing clean clone from GitHub

if Path("/kaggle/working").is_dir():
    workspace = Path("/kaggle/working")
elif Path("/content").is_dir():
    workspace = Path("/content")
else:
    workspace = Path.cwd()

repo_dir = workspace / REPO_FOLDER
project_dir = repo_dir / PROJECT_FOLDER

# Local development support: use the existing source folder outside Kaggle/Colab.
local_project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "project_utils.py").is_file()), None)
if local_project is not None and not (Path("/kaggle/working").is_dir() or Path("/content").is_dir()):
    ROOT = local_project
else:
    if not repo_dir.exists():
        print("Cloning:", REPO_URL)
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
    elif PULL_LATEST:
        # Only fast-forward; protects existing local changes from being overwritten.
        subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only"])
    else:
        print("Using existing clone:", repo_dir)

    ROOT = project_dir

if not (ROOT / "project_utils.py").is_file():
    raise FileNotFoundError(
        f"Cannot find project_utils.py at {ROOT}. Confirm the GitHub folder name."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
print("Project loaded from:", ROOT)


## A. Environment and safety checks

Run this notebook **first**. The first three cells work without downloading Qwen. If packages are missing, run the installation cell in Kaggle/Colab (restart the kernel once after installation). This notebook does not generate paper results.

In [ ]:
import sys, platform, importlib.util
print('Python:', sys.version.split()[0], 'OS:', platform.platform())
for pkg in ['torch','transformers','peft','accelerate','datasets','sklearn','numpy','pandas']:
    spec = importlib.util.find_spec(pkg)
    print(f'{pkg:15}', 'installed' if spec else 'MISSING')
try:
    import torch
    print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0)); print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
except ImportError:
    print('Install PyTorch in Kaggle/Colab using the platform-default build.')

### Verify dependencies before training

A different Transformers version can change TrainingArguments names. The project now checks both APIs; however, the tested project dependency set remains `transformers==4.51.3`, `peft==0.15.2`, and `accelerate==1.6.0`. If Kaggle has different packages, enable installation below and **restart the session** before any imports or GPU training. Do not uninstall or replace the preinstalled CUDA PyTorch.


In [ ]:
# Reinstall the pinned project dependencies if versions differ. Restart session afterward.
# Do NOT install torch again; keep the Kaggle/Colab CUDA build.
INSTALL_PACKAGES = False
if INSTALL_PACKAGES:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT/'requirements.txt')], check=True)
    print('RESTART THE KAGGLE/COLAB SESSION. Then run cells from the top.')


In [ ]:
# Offline unit tests: no model download or GPU required.
import subprocess
command = [sys.executable, '-m', 'pytest', '-q', str(ROOT/'tests')]
res = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
print(res.stdout[-4000:], res.stderr[-1000:]); assert res.returncode == 0, 'Unit tests failed'

In [ ]:
# API preflight: no Qwen download and no GPU training.
# This catches removed/renamed TrainingArguments options in one check.
from project_utils import check_training_api, train_run
try:
    api = check_training_api()
    print('Training API ready:')
    for name, value in api.items():
        if name != 'training_arguments':
            print(f'  {name}: {value}')
except Exception as exc:
    print('Training API preflight FAILED:', type(exc).__name__, str(exc))
    print('Install the pinned requirements, restart session, or update library compatibility.')
    raise


## B. Optional tokenizer and model smoke test

Set `RUN_MODEL_SMOKE=True` only once packages are installed and internet is available. This loads the real 0.5B model and checks that exactly the answer tokens receive loss.

In [ ]:
RUN_MODEL_SMOKE = False
if RUN_MODEL_SMOKE:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(cfg()['model_id'])
    labels = DEFAULT_LABELS
    example = encode_supervised(tokenizer, 'پاکستان نے کرکٹ میچ جیت لیا', 'sports', labels, 'ur', max_length=256)
    assert len(example['input_ids']) == len(example['labels'])
    assert any(x != -100 for x in example['labels'])
    print('Prompt tokens ignored:', sum(x == -100 for x in example['labels']))
    print('Supervised tokens:',sum(x != -100 for x in example['labels']))
    model, tok = load_model(cfg()['model_id'])
    import torch
    with torch.no_grad():
        batch = supervised_collator(tok)([example]); batch = {k:v.to(next(model.parameters()).device) for k,v in batch.items()}
        loss = model(**batch).loss
    print('Real model loss:',float(loss)); assert torch.isfinite(loss).item()
    del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None

## C. Optional two-step GPU training pilot

Run `01_preprocess_data` first. **This is not a scientific result**: it measures real training speed and checks whether full fine-tuning fits. Test LoRA first, then full. Stop and reduce `config.json → max_length` to 192 or 128 if full training runs out of GPU memory. Use two training steps, not an entire epoch. Both pilots use the **same Adafactor optimizer**, so the optimizer is not a hidden difference.

In [ ]:
RUN_GPU_PILOT = False  # Set True after 01_preprocess_data and GPU enabled
if RUN_GPU_PILOT:
    import torch
    assert torch.cuda.is_available(), 'GPU required for timing pilot'
    for method in ['lora','full']:
        try:
            r = train_run(method, 'sib200', size=5, seed=42, rank=8, max_steps=2, pilot=True,
                          run_name=f'pilot_{method}_{int(__import__("time").time())}')
            print(method, '2 steps:', r['wall_seconds'], 'seconds;', 'VRAM GiB:', r['peak_gpu_allocated_gib'])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f'{method}: OOM. Reduce sequence length / keep batch=1, or try another available GPU.')

## D. Runtime estimation after pilot

A rough estimate from a two-step run can be misleading because it includes model loading, initialization, and warmup. Prefer a 10–20-step pilot for better numbers. No result should be reported as a measured full-training time until that training actually completes.

In [ ]:
pilots = [r for r in read_jsonl(ROOT/'outputs/metrics/train_runs.jsonl') if r.get('pilot')]
for r in pilots:
    print(r['run_id'], 'steps=',r['global_steps'], 'total wall min=',round(r['wall_seconds']/60,1),
          'avg sec/step incl. setup=',round(r['wall_seconds']/max(1,r['global_steps']),2))